In [1]:

!pip install altair pandas vl-convert-python pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 50.2 MB/s eta 0:00:00


In [20]:

import pandas as pd
from google.colab import files

uploaded = files.upload()

df = pd.read_csv(
    next(iter(uploaded)),
    sep=';',
    encoding='cp1252',
    skiprows=2,
    decimal=','
)

df.head()


Saving Valor Neto Premier League..csv to Valor Neto Premier League. (4).csv


,Año,Arsenal (€),Chelsea (€),Everton (€),Liverpool (€),Manchester United (€),Tottenham Hotspur (€),Manchester City (€),Newcastle United (€),Aston Villa (€),...,Southampton (€),Leicester City (€),Crystal Palace (€),Sunderland (€),West Bromwich Albion (€),Wolverhampton Wanderers (€),Blackburn Rovers (€),Bolton Wanderers (€),Middlesbrough (€),Stoke City (€)
0,2005,240,250,75.9,181.2,246.4,104.5,90.1,128.9,64.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78.6,77.0,NaN
1,2006,464,381,75.0,176.0,242.6,107.2,89.4,124.3,62.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2007,600,578,98.0,198.9,315.2,153.1,84.6,129.4,62.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2008,598,381,98.0,210.9,324.8,145.0,104.0,126.6,78.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009,605,"366,6",143.0,237.0,327.0,132.7,101.2,101.0,95.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:

# Reconstruir los datos desde la base original
datos = df.melt(
    id_vars='Año',
    var_name='Club',
    value_name='Valor'
)

# Limpiar los nombres de los clubes
datos['Club'] = datos['Club'].str.replace(
    ' (€)', '', regex=False
).str.strip()

# Eliminar solamente los valores vacíos
datos = datos.dropna(subset=['Valor'])

# Comprobar que ahora aparecen todos
print("Cantidad de clubes:", datos['Club'].nunique())
print(datos['Club'].unique())


Cantidad de clubes: 21
['Arsenal' 'Chelsea' 'Everton' 'Liverpool' 'Manchester United'
 'Tottenham Hotspur' 'Manchester City' 'Newcastle United' 'Aston Villa'
 'West Ham United' 'Fulham' 'Southampton' 'Leicester City'
 'Crystal Palace' 'Sunderland' 'West Bromwich Albion'
 'Wolverhampton Wanderers' 'Blackburn Rovers' 'Bolton Wanderers'
 'Middlesbrough' 'Stoke City']


In [19]:
import altair as alt

# Convertir los valores a números
for columna in df.columns[1:]:
    df[columna] = pd.to_numeric(
        df[columna].astype(str)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False),
        errors='coerce'
    )

# Pasar de formato ancho a largo
datos = df.melt(
    id_vars='Año',
    var_name='Club',
    value_name='Valor'
)

# Limpiar nombres
datos['Club'] = datos['Club'].str.replace(
    ' (€)', '', regex=False
).str.strip()

# Seleccionar clubes
clubes = [
    'Arsenal',
    'Chelsea',
    'Liverpool',
    'Manchester United',
    'Manchester City'
]

datos = datos[datos['Club'].isin(clubes)]
datos = datos.dropna(subset=['Valor'])

# Crear visualización corregida
grafico = alt.Chart(datos).mark_line(
    point=True,
    strokeWidth=3
).encode(
    x=alt.X('Año:Q', title='Año',
            scale=alt.Scale(domain=[2005, 2025])),
    y=alt.Y('Valor:Q', title='Valor registrado (€)'),
    color=alt.Color('Club:N', title='Club'),
    tooltip=[
        'Club:N',
        'Año:Q',
        alt.Tooltip('Valor:Q', format=',.1f')
    ]
).properties(
    title=alt.TitleParams(
        text='La evolución económica de los gigantes ingleses',
        subtitle='Comparación entre cinco clubes, 2005–2025'
    ),
    width=750,
    height=420
).interactive()

grafico

alt.Chart(...)

In [21]:

import pandas as pd
import altair as alt

# Recuperar TODOS los clubes desde la base original
datos = df.melt(
    id_vars='Año',
    var_name='Club',
    value_name='Valor'
)

# Limpiar nombres y valores
datos['Club'] = datos['Club'].str.replace(
    ' (€)', '', regex=False
).str.strip()

datos['Valor'] = pd.to_numeric(
    datos['Valor'], errors='coerce'
)

datos['Año'] = pd.to_numeric(
    datos['Año'], errors='coerce'
)

datos = datos.dropna(subset=['Valor', 'Año'])

# Selección interactiva de clubes
seleccion = alt.selection_point(
    fields=['Club'],
    bind='legend'
)

# Crear gráfico
grafico = alt.Chart(datos).mark_line(
    point=True,
    strokeWidth=2
).encode(
    x=alt.X(
        'Año:Q',
        title='Año',
        scale=alt.Scale(domain=[2005, 2025])
    ),
    y=alt.Y(
        'Valor:Q',
        title='Valor registrado (€)'
    ),
    color=alt.Color(
        'Club:N',
        title='Club'
    ),
    opacity=alt.condition(
        seleccion,
        alt.value(1),
        alt.value(0.12)
    ),
    tooltip=[
        'Club:N',
        'Año:Q',
        alt.Tooltip('Valor:Q', format=',.1f')
    ]
).add_params(
    seleccion
).properties(
    title=alt.TitleParams(
        text='La evolución económica de la Premier League',
        subtitle='Valor de los clubes entre 2005 y 2025'
    ),
    width=850,
    height=500
).interactive()

# Mostrar cantidad de clubes
print("Clubes incluidos:", datos['Club'].nunique())

# Mostrar gráfico
display(grafico)


Clubes incluidos: 21


alt.Chart(...)

In [22]:

import altair as alt

# Selección interactiva desde la leyenda
seleccion = alt.selection_point(
    fields=['Club'],
    bind='legend'
)

grafico = alt.Chart(datos).mark_line(
    point=True,
    strokeWidth=2
).encode(
    x=alt.X(
        'Año:Q',
        title='Año',
        scale=alt.Scale(domain=[2005, 2025])
    ),
    y=alt.Y(
        'Valor:Q',
        title='Valor registrado (€)'
    ),
    color=alt.Color(
        'Club:N',
        title='Club'
    ),
    opacity=alt.condition(
        seleccion,
        alt.value(1),
        alt.value(0.12)
    ),
    tooltip=[
        'Club:N',
        'Año:Q',
        alt.Tooltip('Valor:Q', format=',.1f')
    ]
).add_params(
    seleccion
).properties(
    title=alt.TitleParams(
        text='20 años de evolución económica en la Premier League',
        subtitle='Evolución del valor de los clubes entre 2005 y 2025'
    ),
    width=850,
    height=500
).interactive()

grafico


alt.Chart(...)

In [23]:

from google.colab import files
from PIL import Image

# Guardar gráfico interactivo en HTML
grafico.save('premier_league.html')

# Guardar gráfico como imagen
grafico.save('premier_league.png', scale_factor=2)

# Convertir imagen a JPG
imagen = Image.open('premier_league.png')
imagen.convert('RGB').save(
    'premier_league.jpg',
    quality=95
)

# Descargar ambos archivos
files.download('premier_league.html')
files.download('premier_league.jpg')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>